In [1]:
"""
Loops through ROIs on a given OMERO image. For each ROI shape:
  - Skips if a comment already exists
  - Matches the shape's RGB fill color to a preset palette (±10 tolerance per channel)
  - Adds the matching annotation label as a comment if a match is found
 
Usage:
    1. Fill in your credentials and settings in the CONFIG section below
    2. Run:  python omero_roi_comment.py
"""

import omero
import omero.clients
from omero.gateway import BlitzGateway
from omero.rtypes import rstring
import os
import csv
import sys
import pandas as pd
from datetime import datetime
from pathlib import Path

# Configuration
OMERO_HOST = "wss://wsi.lavlab.mcw.edu/omero-wss"
OMERO_PORT = 443
OMERO_USER = "sduenweg"
OMERO_PASSWORD = "Jasper100308#"

LOG_DIR = "/Volumes/Siren/Prostate_data/SD_Pathomics"

In [2]:
# Define Omero Color Key
COLOR_PALETTE = {
    (25,20,255): "Seminal Vesicles",
    (0,0,0): "Atrophy",
    (255,122,0): "HGPIN",
    (255,16,0): "Exclusion ROI",
    (48,255,50): "G3",
    (255,250,20): "G4NC",
    (254,22,255): "G4CG",
    (33,255,255): "G5",
    (128,128,128): "Urethra",
}

COLOR_TOLERANCE = 10 # ± tolerance for color matching in ROI detection

In [3]:
def int_to_rgba(color_int: int) -> tuple[int, int, int, int]:
    """Convert OMERO's packed RGBA integer to (r, g, b, a)."""
    # OMERO stores color as a signed 32-bit int: RRGGBBAA
    if color_int < 0:
        color_int += 2**32
    r = (color_int >> 24) & 0xFF
    g = (color_int >> 16) & 0xFF
    b = (color_int >> 8)  & 0xFF
    a = color_int         & 0xFF
    return r, g, b, a
 
 
def match_color(r: int, g: int, b: int) -> str | None:
    """Return the palette label whose RGB is within tolerance, or None."""
    for (pr, pg, pb), label in COLOR_PALETTE.items():
        if (abs(r - pr) <= COLOR_TOLERANCE and
                abs(g - pg) <= COLOR_TOLERANCE and
                abs(b - pb) <= COLOR_TOLERANCE):
            return label
    return None

def get_shape_color(shape) -> tuple[int, int, int] | None:
    """Extract (r, g, b) from a shape's fill or stroke color."""
    # Prefer fill color; fall back to stroke color
    color_val = None
    if shape.getStrokeColor() is not None:
        color_val = shape.getStrokeColor().getValue()
 
    if color_val is None:
        return None
 
    r, g, b, _ = int_to_rgba(color_val)
    return r, g, b
 
 
def shape_has_comment(shape) -> bool:
    """Return True if the shape already has a non-empty textValue (comment)."""
    tv = shape.getTextValue()
    if tv is None:
        return False
    return bool(tv.getValue().strip())

In [8]:
def process_image(conn: BlitzGateway, image_id: int):
    image = conn.getObject("Image", image_id)
    if image is None:
        print(f"ERROR: Image {image_id} not found or not accessible.")
        sys.exit(1)
 
    image_name = image.getName()
    print(f"\nImage: [{image_id}] {image_name}")
 
    roi_service = conn.getRoiService()
    result = roi_service.findByImage(image_id, None)
    rois = result.rois
 
    # print(f"  Found {len(rois)} ROI(s)\n")
 
    updated = 0
    skipped_has_comment = 0
    skipped_no_match = 0
 
    for roi in rois:
        roi_id = roi.getId().getValue()
        shapes = roi.copyShapes()
 
        for shape in shapes:
            shape_type = shape.__class__.__name__
 
            # --- Skip if already has a comment ---
            if shape_has_comment(shape):
                skipped_has_comment += 1
                continue
 
            # --- Get color ---
            rgb = get_shape_color(shape)
            if rgb is None:
                skipped_no_match += 1
                continue
 
            r, g, b = rgb
 
            # --- Match to palette ---
            label = match_color(r, g, b)
            if label is None:
                skipped_no_match += 1
    
                continue

            shape.setTextValue(rstring(label))
            conn.getUpdateService().saveObject(shape)
            updated += 1

    # print(f"\n  Summary:")
    # print(f"    Updated  : {updated}")
    # print(f"    Skipped (had comment)  : {skipped_has_comment}")
    # print(f"    Skipped (no color match): {skipped_no_match}")

    return {
        "IMAGE_ID":     image_id,
        "NAME":         image_name,
        "NUM_FOUND":    len(rois),
        "NUM_UPDATED":  updated,
        "NUM_SKIP":     skipped_has_comment,
        "NUM_NO_MATCH": skipped_no_match,
    }
 

In [ ]:
print(f"Connecting to OMERO at {OMERO_HOST}:{OMERO_PORT} as '{OMERO_USER}' ...")
conn = BlitzGateway(OMERO_USER, OMERO_PASSWORD, host=OMERO_HOST, port=OMERO_PORT, secure=True)
conn.connect()
conn.c.enableKeepAlive(60)

# Load the list of image IDs to process in omero_ids.txt
with open("omero_ids.txt", "r") as f:
    IMAGE_IDS = [int(line.strip()) for line in f.readlines()]


log_rows = []
for IMAGE_ID in IMAGE_IDS:
    row = process_image(conn, IMAGE_ID)
    log_rows.append(row)
 

filename = datetime.now().strftime("omero_roi_comments-%m_%d_%y.csv")
df = pd.DataFrame(log_rows, columns=["IMAGE_ID", "NAME", "NUM_FOUND", "NUM_UPDATED", "NUM_SKIP", "NUM_NO_MATCH"])
df.to_csv(filename, index=False)
